# Load the SegRap2023 Dataset

In [1]:
# For those who have the folder as a shortcut as opposed to owning

from google.colab import drive
drive.mount('/content/drive')

!pip install -q nibabel

from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Biomedical Data Design')
DATA = ROOT / 'SegRap2023_Training_Set_120cases'
OUT = ROOT / 'outputs'

# Create outputs folder if it does not already exist
OUT.mkdir(parents=True, exist_ok=True)

# Check that the dataset shortcut/path is accessible
if not DATA.exists():
    raise FileNotFoundError(
        f"Dataset not found at:\n{DATA}\n\n"
        "Make sure the shared folder has been added as a shortcut to My Drive "
        "and that ROOT matches the shortcut location."
    )

cases = sorted(p.name for p in DATA.glob('segrap_*'))

print(f'{len(cases)} cases found, e.g. {cases[:3]}')
print('Example case contents:',
      len(list((DATA / cases[0]).glob('*.nii.gz'))),
      'files')
print('Outputs go to:', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
120 cases found, e.g. ['segrap_0000', 'segrap_0001', 'segrap_0002']
Example case contents: 49 files
Outputs go to: /content/drive/MyDrive/Biomedical Data Design/outputs


# Load the SegRap2023 Dataset

In [2]:
# Step 1A: Inspect the files contained in one SegRap2023 patient folder

from pathlib import Path

# Pick one example case
case = DATA / cases[0]

print("Case:", case.name)
print("\nFiles:")

for f in sorted(case.glob("*.nii.gz")):
    print(" -", f.name)

Case: segrap_0000

Files:
 - Brain.nii.gz
 - BrainStem.nii.gz
 - Chiasm.nii.gz
 - Cochlea_L.nii.gz
 - Cochlea_R.nii.gz
 - ETbone_L.nii.gz
 - ETbone_R.nii.gz
 - Esophagus.nii.gz
 - Eye_L.nii.gz
 - Eye_R.nii.gz
 - GTVnd.nii.gz
 - GTVp.nii.gz
 - Hippocampus_L.nii.gz
 - Hippocampus_R.nii.gz
 - IAC_L.nii.gz
 - IAC_R.nii.gz
 - Larynx.nii.gz
 - Larynx_Glottic.nii.gz
 - Larynx_Supraglot.nii.gz
 - Lens_L.nii.gz
 - Lens_R.nii.gz
 - Mandible_L.nii.gz
 - Mandible_R.nii.gz
 - Mastoid_L.nii.gz
 - Mastoid_R.nii.gz
 - MiddleEar_L.nii.gz
 - MiddleEar_R.nii.gz
 - OpticNerve_L.nii.gz
 - OpticNerve_R.nii.gz
 - OralCavity.nii.gz
 - Parotid_L.nii.gz
 - Parotid_R.nii.gz
 - PharynxConst.nii.gz
 - Pituitary.nii.gz
 - SpinalCord.nii.gz
 - Submandibular_L.nii.gz
 - Submandibular_R.nii.gz
 - TMjoint_L.nii.gz
 - TMjoint_R.nii.gz
 - TemporalLobe_L.nii.gz
 - TemporalLobe_R.nii.gz
 - Thyroid.nii.gz
 - Trachea.nii.gz
 - TympanicCavity_L.nii.gz
 - TympanicCavity_R.nii.gz
 - VestibulSemi_L.nii.gz
 - VestibulSemi_R.nii.g

In [3]:
# Step 1B: Check shape, voxel spacing, and affine alignment for one CT and one segmentation mask

import nibabel as nib
import numpy as np

ct_path = case / "image.nii.gz"
mask_path = case / "Brain.nii.gz"

ct_img = nib.load(ct_path)
mask_img = nib.load(mask_path)

print("CT shape:", ct_img.shape)
print("Mask shape:", mask_img.shape)

print("\nCT voxel spacing:", ct_img.header.get_zooms())
print("Mask voxel spacing:", mask_img.header.get_zooms())

print("\nAffine matrices match:",
      np.allclose(ct_img.affine, mask_img.affine))

CT shape: (1024, 1024, 127)
Mask shape: (1024, 1024, 127)

CT voxel spacing: (np.float32(0.53222656), np.float32(0.53222656), np.float32(3.0))
Mask voxel spacing: (np.float32(0.53222656), np.float32(0.53222656), np.float32(3.0))

Affine matrices match: True


In [4]:
# Step 1C: Verify that every segmentation mask in this patient folder matches the CT geometry

mask_files = [
    f for f in sorted(case.glob("*.nii.gz"))
    if f.name not in ["image.nii.gz", "image_contrast.nii.gz"]
]

mismatches = []

for mask_path in mask_files:
    mask_img = nib.load(mask_path)

    same_shape = mask_img.shape == ct_img.shape
    same_spacing = np.allclose(mask_img.header.get_zooms(), ct_img.header.get_zooms())
    same_affine = np.allclose(mask_img.affine, ct_img.affine)

    if not (same_shape and same_spacing and same_affine):
        mismatches.append({
            "file": mask_path.name,
            "same_shape": same_shape,
            "same_spacing": same_spacing,
            "same_affine": same_affine
        })

print("Total masks checked:", len(mask_files))
print("Mismatches found:", len(mismatches))

if mismatches:
    for m in mismatches:
        print(m)
else:
    print("All masks match the CT geometry.")

Total masks checked: 47
Mismatches found: 0
All masks match the CT geometry.


In [5]:
# Step 1D: Check that all segmentation masks in this patient are binary

unique_values_summary = {}

for mask_path in mask_files:
    mask_img = nib.load(mask_path)
    mask_data = mask_img.get_fdata()

    unique_vals = np.unique(mask_data)
    unique_values_summary[mask_path.name] = unique_vals

non_binary = {
    name: vals
    for name, vals in unique_values_summary.items()
    if not np.all(np.isin(vals, [0, 1]))
}

print("Total masks checked:", len(mask_files))
print("Non-binary masks found:", len(non_binary))

if non_binary:
    for name, vals in non_binary.items():
        print(name, vals[:20])
else:
    print("All masks are binary.")

Total masks checked: 47
Non-binary masks found: 47
Brain.nii.gz [  0. 255.]
BrainStem.nii.gz [  0. 255.]
Chiasm.nii.gz [  0. 255.]
Cochlea_L.nii.gz [  0. 255.]
Cochlea_R.nii.gz [  0. 255.]
ETbone_L.nii.gz [  0. 255.]
ETbone_R.nii.gz [  0. 255.]
Esophagus.nii.gz [  0. 255.]
Eye_L.nii.gz [  0. 255.]
Eye_R.nii.gz [  0. 255.]
GTVnd.nii.gz [  0. 255.]
GTVp.nii.gz [  0. 255.]
Hippocampus_L.nii.gz [  0. 255.]
Hippocampus_R.nii.gz [  0. 255.]
IAC_L.nii.gz [  0. 255.]
IAC_R.nii.gz [  0. 255.]
Larynx.nii.gz [  0. 255.]
Larynx_Glottic.nii.gz [  0. 255.]
Larynx_Supraglot.nii.gz [  0. 255.]
Lens_L.nii.gz [  0. 255.]
Lens_R.nii.gz [  0. 255.]
Mandible_L.nii.gz [  0. 255.]
Mandible_R.nii.gz [  0. 255.]
Mastoid_L.nii.gz [  0. 255.]
Mastoid_R.nii.gz [  0. 255.]
MiddleEar_L.nii.gz [  0. 255.]
MiddleEar_R.nii.gz [  0. 255.]
OpticNerve_L.nii.gz [  0. 255.]
OpticNerve_R.nii.gz [  0. 255.]
OralCavity.nii.gz [  0. 255.]
Parotid_L.nii.gz [  0. 255.]
Parotid_R.nii.gz [  0. 255.]
PharynxConst.nii.gz [  0. 255.]

In [6]:
# Step 1E: Compare non-contrast and contrast-enhanced CT geometry for this patient

cect_path = case / "image_contrast.nii.gz"
cect_img = nib.load(cect_path)

print("Non-contrast CT shape:", ct_img.shape)
print("Contrast CT shape:", cect_img.shape)

print("\nNon-contrast spacing:", ct_img.header.get_zooms())
print("Contrast spacing:", cect_img.header.get_zooms())

print("\nAffine matrices match:",
      np.allclose(ct_img.affine, cect_img.affine))

Non-contrast CT shape: (1024, 1024, 127)
Contrast CT shape: (1024, 1024, 127)

Non-contrast spacing: (np.float32(0.53222656), np.float32(0.53222656), np.float32(3.0))
Contrast spacing: (np.float32(0.53222656), np.float32(0.53222656), np.float32(3.0))

Affine matrices match: True


In [7]:
# Step 1F: Check image-mask geometry consistency across the first 10 patients

num_cases_to_check = 10
geometry_issues = []

for case_name in cases[:num_cases_to_check]:
    case_path = DATA / case_name

    ct_path = case_path / "image.nii.gz"
    ct_img = nib.load(ct_path)

    patient_masks = [
        f for f in case_path.glob("*.nii.gz")
        if f.name not in ["image.nii.gz", "image_contrast.nii.gz"]
    ]

    for mask_path in patient_masks:
        mask_img = nib.load(mask_path)

        same_shape = mask_img.shape == ct_img.shape
        same_spacing = np.allclose(
            mask_img.header.get_zooms(),
            ct_img.header.get_zooms()
        )
        same_affine = np.allclose(
            mask_img.affine,
            ct_img.affine
        )

        if not (same_shape and same_spacing and same_affine):
            geometry_issues.append({
                "case": case_name,
                "mask": mask_path.name,
                "same_shape": same_shape,
                "same_spacing": same_spacing,
                "same_affine": same_affine
            })

print("Patients checked:", num_cases_to_check)
print("Geometry issues found:", len(geometry_issues))

if geometry_issues:
    for issue in geometry_issues[:20]:
        print(issue)
else:
    print("All checked masks match their CT geometry.")

Patients checked: 10
Geometry issues found: 0
All checked masks match their CT geometry.


In [8]:
# Step 1G: Summarize scan shapes and voxel spacing across the first 10 patients

scan_summary = []

for case_name in cases[:10]:
    case_path = DATA / case_name
    img = nib.load(case_path / "image.nii.gz")

    scan_summary.append({
        "case": case_name,
        "shape": img.shape,
        "spacing": tuple(float(x) for x in img.header.get_zooms())
    })

for item in scan_summary:
    print(
        item["case"],
        "| shape:", item["shape"],
        "| spacing:", item["spacing"]
    )

segrap_0000 | shape: (1024, 1024, 127) | spacing: (0.5322265625, 0.5322265625, 3.0)
segrap_0001 | shape: (1024, 1024, 137) | spacing: (0.599609375, 0.599609375, 3.0)
segrap_0002 | shape: (1024, 1024, 121) | spacing: (0.5693359375, 0.5693359375, 3.0)
segrap_0003 | shape: (1024, 1024, 120) | spacing: (0.5107421875, 0.5107421875, 3.0)
segrap_0004 | shape: (1024, 1024, 129) | spacing: (0.5576171875, 0.5576171875, 3.0)
segrap_0005 | shape: (1024, 1024, 168) | spacing: (0.5556640625, 0.5556640625, 3.0)
segrap_0006 | shape: (1024, 1024, 121) | spacing: (0.6171875, 0.6171875, 3.0)
segrap_0007 | shape: (1024, 1024, 130) | spacing: (0.515625, 0.515625, 3.0)
segrap_0008 | shape: (1024, 1024, 115) | spacing: (0.5224609375, 0.5224609375, 3.0)
segrap_0009 | shape: (1024, 1024, 133) | spacing: (0.478515625, 0.478515625, 3.0)


# Validate Image and Annotation Consistency

In [10]:
# Step 2A: Check whether anatomical masks overlap within one patient

import numpy as np
import nibabel as nib

# Reset to one specific patient so CT and masks come from the same case
case = DATA / "segrap_0000"

ct_img = nib.load(case / "image.nii.gz")

mask_files = [
    f for f in sorted(case.glob("*.nii.gz"))
    if f.name not in ["image.nii.gz", "image_contrast.nii.gz"]
]

# Create an array matching this patient's CT dimensions
overlap_count = np.zeros(ct_img.shape, dtype=np.uint8)

for mask_path in mask_files:
    mask_img = nib.load(mask_path)
    mask_data = mask_img.get_fdata()

    # Convert 0/255 segmentation masks into binary 0/1 masks
    binary_mask = mask_data > 0

    overlap_count += binary_mask.astype(np.uint8)

print("Case:", case.name)
print("CT shape:", ct_img.shape)
print("Masks checked:", len(mask_files))

print("\nMaximum number of overlapping labels at one voxel:",
      overlap_count.max())

print("Voxels with at least one label:",
      np.sum(overlap_count >= 1))

print("Voxels with multiple labels:",
      np.sum(overlap_count > 1))

print("Percentage of labeled voxels that overlap:",
      100 * np.sum(overlap_count > 1) / np.sum(overlap_count >= 1))

Case: segrap_0000
CT shape: (1024, 1024, 127)
Masks checked: 47

Maximum number of overlapping labels at one voxel: 3
Voxels with at least one label: 2650010
Voxels with multiple labels: 380400
Percentage of labeled voxels that overlap: 14.35466281259316


In [11]:
# Step 2B: Identify pairs of anatomical masks that overlap

overlapping_pairs = []

mask_data_cache = {}

for mask_path in mask_files:
    data = nib.load(mask_path).get_fdata() > 0
    mask_data_cache[mask_path.stem.replace(".nii", "")] = data

mask_names = list(mask_data_cache.keys())

for i in range(len(mask_names)):
    for j in range(i + 1, len(mask_names)):
        name1 = mask_names[i]
        name2 = mask_names[j]

        overlap_voxels = np.sum(
            mask_data_cache[name1] & mask_data_cache[name2]
        )

        if overlap_voxels > 0:
            overlapping_pairs.append(
                (name1, name2, int(overlap_voxels))
            )

overlapping_pairs.sort(key=lambda x: x[2], reverse=True)

print("Number of overlapping structure pairs:",
      len(overlapping_pairs))

print("\nTop overlapping pairs:")
for pair in overlapping_pairs[:20]:
    print(pair)

Number of overlapping structure pairs: 92

Top overlapping pairs:
('Brain', 'TemporalLobe_L', 135781)
('Brain', 'TemporalLobe_R', 117100)
('Brain', 'BrainStem', 41235)
('Larynx', 'Trachea', 10459)
('GTVp', 'PharynxConst', 10360)
('Larynx', 'PharynxConst', 9626)
('Brain', 'GTVp', 9500)
('Larynx', 'Larynx_Supraglot', 9359)
('Larynx', 'Larynx_Glottic', 6586)
('Brain', 'Hippocampus_L', 6101)
('GTVnd', 'Parotid_L', 5952)
('Hippocampus_L', 'TemporalLobe_L', 5721)
('GTVp', 'OralCavity', 5439)
('Brain', 'Hippocampus_R', 4662)
('Hippocampus_R', 'TemporalLobe_R', 4084)
('GTVnd', 'Submandibular_L', 3107)
('Esophagus', 'Trachea', 2742)
('GTVp', 'TemporalLobe_L', 2487)
('GTVnd', 'Submandibular_R', 2317)
('Brain', 'Chiasm', 1574)


In [12]:
# Step 2C: Verify that all patients contain the same annotation files

reference_masks = set(
    f.name for f in mask_files
)

cases_with_differences = []

for case_name in cases:
    case_path = DATA / case_name

    patient_masks = set(
        f.name for f in case_path.glob("*.nii.gz")
        if f.name not in ["image.nii.gz", "image_contrast.nii.gz"]
    )

    if patient_masks != reference_masks:
        missing = reference_masks - patient_masks
        extra = patient_masks - reference_masks

        cases_with_differences.append({
            "case": case_name,
            "missing": sorted(missing),
            "extra": sorted(extra)
        })

print("Total patients checked:", len(cases))
print("Patients with different mask files:",
      len(cases_with_differences))

if cases_with_differences:
    for issue in cases_with_differences[:10]:
        print(issue)
else:
    print("All patients contain the same 47 annotation files.")

Total patients checked: 120
Patients with different mask files: 12
{'case': 'segrap_0003', 'missing': [], 'extra': ['Pituitary (1).nii.gz']}
{'case': 'segrap_0010', 'missing': [], 'extra': ['Eye_L (1).nii.gz', 'TympanicCavity_L (1).nii.gz']}
{'case': 'segrap_0019', 'missing': [], 'extra': ['Submandibular_L (1).nii.gz']}
{'case': 'segrap_0035', 'missing': [], 'extra': ['TMjoint_L (1).nii.gz']}
{'case': 'segrap_0045', 'missing': [], 'extra': ['Hippocampus_R (1).nii.gz']}
{'case': 'segrap_0057', 'missing': [], 'extra': ['Thyroid (1).nii.gz']}
{'case': 'segrap_0059', 'missing': [], 'extra': ['IAC_L (1).nii.gz']}
{'case': 'segrap_0066', 'missing': [], 'extra': ['BrainStem (1).nii.gz']}
{'case': 'segrap_0068', 'missing': [], 'extra': ['Eye_L (1).nii.gz', 'Parotid_R (1).nii.gz']}
{'case': 'segrap_0072', 'missing': [], 'extra': ['TympanicCavity_L (1).nii.gz']}


In [14]:
# Step 2D: Check for empty segmentation masks in the first 5 patients

import nibabel as nib
import numpy as np

five_cases = cases[:5]

empty_masks = []

for case_name in five_cases:
    case_path = DATA / case_name

    patient_masks = [
        f for f in case_path.glob("*.nii.gz")
        if f.name not in ["image.nii.gz", "image_contrast.nii.gz"]
        and " (1)" not in f.name
    ]

    for mask_path in patient_masks:
        mask_img = nib.load(mask_path)

        # Load the stored mask values without converting to float64
        mask_data = np.asanyarray(mask_img.dataobj)

        if not np.any(mask_data):
            empty_masks.append((case_name, mask_path.name))

print("Patients checked:", len(five_cases))
print("Total empty masks found:", len(empty_masks))

for item in empty_masks:
    print(item)

Patients checked: 5
Total empty masks found: 0


In [15]:
# Step 2E: Check duplicate annotation files in the first 5 patients

duplicate_results = []

for case_name in five_cases:
    case_path = DATA / case_name

    duplicate_files = list(case_path.glob("* (1).nii.gz"))

    for duplicate_path in duplicate_files:
        original_name = duplicate_path.name.replace(" (1).nii.gz", ".nii.gz")
        original_path = case_path / original_name

        if original_path.exists():
            original_data = np.asanyarray(
                nib.load(original_path).dataobj
            )

            duplicate_data = np.asanyarray(
                nib.load(duplicate_path).dataobj
            )

            identical = np.array_equal(
                original_data,
                duplicate_data
            )

            duplicate_results.append({
                "case": case_name,
                "original": original_name,
                "duplicate": duplicate_path.name,
                "identical": identical
            })

for result in duplicate_results:
    print(result)

print("\nTotal duplicate files checked:", len(duplicate_results))
print(
    "Identical duplicates:",
    sum(r["identical"] for r in duplicate_results)
)

{'case': 'segrap_0003', 'original': 'Pituitary.nii.gz', 'duplicate': 'Pituitary (1).nii.gz', 'identical': True}

Total duplicate files checked: 1
Identical duplicates: 1


# Convert the 5-Patient Subset into nnU-Net Format

In [16]:
# Step 3A: Define the five patients used for the initial nnU-Net baseline

five_cases = [
    "segrap_0000",
    "segrap_0001",
    "segrap_0002",
    "segrap_0003",
    "segrap_0004"
]

print("Initial training subset:")
for case_name in five_cases:
    print(case_name)

Initial training subset:
segrap_0000
segrap_0001
segrap_0002
segrap_0003
segrap_0004


In [17]:
# Step 3B: Define a small non-overlapping label set for the initial pipeline test

target_labels = {
    "background": 0,
    "BrainStem": 1,
    "Eye_L": 2,
    "Eye_R": 3,
    "Parotid_L": 4,
    "Parotid_R": 5,
    "SpinalCord": 6
}

print("Target labels:")
for name, label_id in target_labels.items():
    print(label_id, name)

Target labels:
0 background
1 BrainStem
2 Eye_L
3 Eye_R
4 Parotid_L
5 Parotid_R
6 SpinalCord


In [18]:
# Step 3C: Create the nnU-Net dataset directory structure

from pathlib import Path

NNUNET_RAW = Path("/content/nnUNet_raw")
DATASET_NAME = "Dataset501_SegRap5"

dataset_dir = NNUNET_RAW / DATASET_NAME

imagesTr = dataset_dir / "imagesTr"
labelsTr = dataset_dir / "labelsTr"
imagesTs = dataset_dir / "imagesTs"

imagesTr.mkdir(parents=True, exist_ok=True)
labelsTr.mkdir(parents=True, exist_ok=True)
imagesTs.mkdir(parents=True, exist_ok=True)

print("Created:")
print(imagesTr)
print(labelsTr)
print(imagesTs)

Created:
/content/nnUNet_raw/Dataset501_SegRap5/imagesTr
/content/nnUNet_raw/Dataset501_SegRap5/labelsTr
/content/nnUNet_raw/Dataset501_SegRap5/imagesTs


In [19]:
# Step 3D: Copy the five non-contrast CT scans into nnU-Net image format

import shutil

for case_name in five_cases:

    source = DATA / case_name / "image.nii.gz"
    destination = imagesTr / f"{case_name}_0000.nii.gz"

    shutil.copy2(source, destination)

    print(f"Copied {case_name}")

print("\nFinished copying training images.")

Copied segrap_0000
Copied segrap_0001
Copied segrap_0002
Copied segrap_0003
Copied segrap_0004

Finished copying training images.


In [20]:
# Step 3E: Combine selected binary masks into one multiclass nnU-Net label volume

import nibabel as nib
import numpy as np

structures = {
    "BrainStem": 1,
    "Eye_L": 2,
    "Eye_R": 3,
    "Parotid_L": 4,
    "Parotid_R": 5,
    "SpinalCord": 6
}

for case_name in five_cases:

    case_path = DATA / case_name

    reference_img = nib.load(case_path / "image.nii.gz")

    combined_label = np.zeros(
        reference_img.shape,
        dtype=np.uint8
    )

    for structure_name, label_id in structures.items():

        mask_img = nib.load(
            case_path / f"{structure_name}.nii.gz"
        )

        mask = np.asanyarray(mask_img.dataobj) > 0

        # Sanity check: these selected structures should not overlap
        if np.any((combined_label > 0) & mask):
            print(
                f"WARNING: overlap detected in {case_name}: "
                f"{structure_name}"
            )

        combined_label[mask] = label_id

    output_img = nib.Nifti1Image(
        combined_label,
        reference_img.affine,
        reference_img.header
    )

    output_path = labelsTr / f"{case_name}.nii.gz"

    nib.save(output_img, output_path)

    print(f"Created label map for {case_name}")

Created label map for segrap_0000
Created label map for segrap_0001
Created label map for segrap_0002
Created label map for segrap_0003
Created label map for segrap_0004


In [21]:
# Step 3F: Verify the nnU-Net image and label files created for the five patients

print("Training images:")
for f in sorted(imagesTr.glob("*.nii.gz")):
    print(" ", f.name)

print("\nTraining labels:")
for f in sorted(labelsTr.glob("*.nii.gz")):
    print(" ", f.name)

print("\nNumber of images:", len(list(imagesTr.glob("*.nii.gz"))))
print("Number of labels:", len(list(labelsTr.glob("*.nii.gz"))))

Training images:
  segrap_0000_0000.nii.gz
  segrap_0001_0000.nii.gz
  segrap_0002_0000.nii.gz
  segrap_0003_0000.nii.gz
  segrap_0004_0000.nii.gz

Training labels:
  segrap_0000.nii.gz
  segrap_0001.nii.gz
  segrap_0002.nii.gz
  segrap_0003.nii.gz
  segrap_0004.nii.gz

Number of images: 5
Number of labels: 5


In [22]:
# Step 3G: Create the nnU-Net dataset.json metadata file

import json

dataset_json = {
    "channel_names": {
        "0": "CT"
    },
    "labels": {
        "background": 0,
        "BrainStem": 1,
        "Eye_L": 2,
        "Eye_R": 3,
        "Parotid_L": 4,
        "Parotid_R": 5,
        "SpinalCord": 6
    },
    "numTraining": 5,
    "file_ending": ".nii.gz"
}

dataset_json_path = dataset_dir / "dataset.json"

with open(dataset_json_path, "w") as f:
    json.dump(dataset_json, f, indent=4)

print("Created:", dataset_json_path)

with open(dataset_json_path, "r") as f:
    print(f.read())

Created: /content/nnUNet_raw/Dataset501_SegRap5/dataset.json
{
    "channel_names": {
        "0": "CT"
    },
    "labels": {
        "background": 0,
        "BrainStem": 1,
        "Eye_L": 2,
        "Eye_R": 3,
        "Parotid_L": 4,
        "Parotid_R": 5,
        "SpinalCord": 6
    },
    "numTraining": 5,
    "file_ending": ".nii.gz"
}


In [23]:
# Step 3H: Verify the converted nnU-Net dataset before preprocessing

for case_name in five_cases:
    image_path = imagesTr / f"{case_name}_0000.nii.gz"
    label_path = labelsTr / f"{case_name}.nii.gz"

    image_img = nib.load(image_path)
    label_img = nib.load(label_path)

    label_data = np.asanyarray(label_img.dataobj)

    print(f"\n{case_name}")
    print("Image shape:", image_img.shape)
    print("Label shape:", label_img.shape)
    print("Geometry match:", np.allclose(image_img.affine, label_img.affine))
    print("Label values:", np.unique(label_data))


segrap_0000
Image shape: (1024, 1024, 127)
Label shape: (1024, 1024, 127)
Geometry match: True
Label values: [0 1 2 3 4 5 6]

segrap_0001
Image shape: (1024, 1024, 137)
Label shape: (1024, 1024, 137)
Geometry match: True
Label values: [0 1 2 3 4 5 6]

segrap_0002
Image shape: (1024, 1024, 121)
Label shape: (1024, 1024, 121)
Geometry match: True
Label values: [0 1 2 3 4 5 6]

segrap_0003
Image shape: (1024, 1024, 120)
Label shape: (1024, 1024, 120)
Geometry match: True
Label values: [0 1 2 3 4 5 6]

segrap_0004
Image shape: (1024, 1024, 129)
Label shape: (1024, 1024, 129)
Geometry match: True
Label values: [0 1 2 3 4 5 6]


# Run nnU-Net Fingerprinting and Preprocessing

In [2]:
# Check whether the nnU-Net dataset folder still exists (crashed before)

from pathlib import Path

dataset_dir = Path("/content/nnUNet_raw/Dataset501_SegRap5")

print("Dataset folder exists:", dataset_dir.exists())

if dataset_dir.exists():
    print("\nContents:")
    for item in sorted(dataset_dir.iterdir()):
        print(" -", item.name)

Dataset folder exists: True

Contents:
 - dataset.json
 - imagesTr
 - imagesTs
 - labelsTr


In [9]:
# Step 4A: Install nnU-Net v2

!pip install -q nnunetv2

In [10]:
# Step 4B: Define nnU-Net raw, preprocessed, and results directories

import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/nnUNet_results


In [11]:
# Step 4C: Extract the dataset fingerprint

!nnUNetv2_extract_fingerprint -d 501

Dataset501_SegRap5


In [12]:
# Step 4D: Generate the nnU-Net experiment plan

!nnUNetv2_plan_experiment -d 501


############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [3.        0.5743457 0.5743457]. 
Current patch size: (np.int64(32), np.int64(256), np.int64(224)). 
Current median shape: [127.         994.17475728 994.17475728]
Attempting to find 3d_lowres config. 
Current spacing: [3.         0.59157607 0.59157607]. 
Current patch size: (np.int64(32), np.int64(256), np.int64(224)). 
Current median shape: [127.         965.21821095 965.21821095]
Attempting to find 3d_lowres config. 
Current spacing: [3.         0.60932336 0.60932336]. 
Current patch size: (np.int64(32), np.int64(256), np.int64(224)). 
Current median shape: [127.         937.10505918 937.10505918]
Attempting to find 3d_lowres config. 
Current spacing: 

In [13]:
# Step 4E: Preprocess only the 3D full-resolution configuration using one worker

!nnUNetv2_preprocess -d 501 -c 3d_fullres -np 1

Preprocessing dataset Dataset501_SegRap5
Configuration: 3d_fullres...
{'data_identifier': 'nnUNetPlans_3d_fullres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': [28, 256, 256], 'median_image_size_in_voxels': [127.0, 1024.0, 1024.0], 'spacing': [3.0, 0.5576171875, 0.5576171875], 'normalization_schemes': ['CTNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.architectures.unet.PlainConvUNet', 'arch_kwargs': {'n_stages': 7,